# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² mass spectrometry dataset using the [mlcroissant](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described using a Croissant schema

> Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
We load the dataset from its Croissant URL, inspect basic metadata, and check for contained record sets.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# URL for the Croissant schema
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
print(dataset.metadata)

## 2. Data Overview

Let's review the available record sets and their fields, referencing each by their `@id`.

The FAIR² dataset defines record sets for protein quantification and processed protein lists. We'll list all record sets and their fields to determine which ones are suitable for analysis.

In [ ]:
# List record sets and fields by @id
from pprint import pprint

record_sets = list(dataset.record_sets)
print("Available record sets in this dataset:")
for rec in record_sets:
    print(f"- RecordSet name: {rec.name}, @id: {rec.id_}")
    print("  Fields:")
    for field in rec.fields:
        print(f"    - {field.name} (@id: {field.id_}, data_type: {field.data_type})")

For illustration, let's preview the records from the protein quantification record set (`cr:RecordSet/protein_quantifications`).

In [ ]:
# Preview records from a specific record set by @id
protein_quant_rs_id = 'cr:RecordSet/protein_quantifications'  # Example ID, update based on overview above
# If you don't know the exact @id, use the output above to select
print(f"First 2 records from record set '{protein_quant_rs_id}':")
for i, record in enumerate(dataset.records(record_set=protein_quant_rs_id)):
    pprint(record)
    if i == 1:
        break

## 3. Data Extraction

Load data from each main record set into pandas DataFrames.

**Note**: The key record sets as per the schema are `cr:RecordSet/protein_quantifications` and `cr:RecordSet/protein_list` (replace with actual `@id`s found above if different).

In [ ]:
# List of selected record set @ids
record_sets_ids = ['cr:RecordSet/protein_quantifications', 'cr:RecordSet/protein_list']
dfs = {}

for rec_id in record_sets_ids:
    all_records = list(dataset.records(record_set=rec_id))
    if all_records:
        dfs[rec_id] = pd.DataFrame(all_records)

print('Fields in protein quantifications table:')
if 'cr:RecordSet/protein_quantifications' in dfs:
    print(dfs['cr:RecordSet/protein_quantifications'].columns.tolist())
    display(dfs['cr:RecordSet/protein_quantifications'].head())

## 4. Exploratory Data Analysis (EDA)

Let's analyze the `coverage_percent` field (if available) in `cr:RecordSet/protein_quantifications`. We'll filter proteins with >10% coverage and normalize their values. Finally, we'll group by sample or condition if a grouping field exists.

Update field `@id`s as appropriate for your specific dataset.

In [ ]:
# Pick a numeric field to analyze
numeric_field = 'coverage_percent'  # This should match the @id or name of your dataset field
rs_id = 'cr:RecordSet/protein_quantifications'

if rs_id in dfs and numeric_field in dfs[rs_id].columns:
    threshold = 10
    df = dfs[rs_id]
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, norm_col]].head())

    # Try grouping by 'sample' or a similar field
    group_field = 'sample_id'  # Update to the actual group @id or column available
    if group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"\nGrouped by {group_field} (first rows):")
        print(grouped_df.head())
else:
    print(f"Field {numeric_field} not found in columns of record set '{rs_id}'. Available fields:")
    print(dfs[rs_id].columns if rs_id in dfs else "No DataFrame available.")

## 5. Visualization
Let's visualize the distribution of coverage percent for proteins with >10% coverage, and (if applicable), the mean normalized coverage by sample or group.

You can modify the visualization fields according to your dataset's column names.

In [ ]:
import matplotlib.pyplot as plt

if rs_id in dfs and numeric_field in dfs[rs_id].columns:
    plt.figure(figsize=(8,4))
    filtered_df[numeric_field].hist(bins=30)
    plt.title("Distribution of Protein Coverage Percent (>10%)")
    plt.xlabel("Coverage Percent")
    plt.ylabel("Count")
    plt.show()

    if 'sample_id' in filtered_df.columns:
        # Plot mean normalized coverage per group
        grouped = filtered_df.groupby('sample_id')[norm_col].mean()
        grouped.plot(kind='bar', figsize=(10,4))
        plt.title("Mean Normalized Coverage by Sample")
        plt.xlabel("Sample ID")
        plt.ylabel("Mean Normalized Coverage")
        plt.show()

## 6. Conclusion
In this notebook, we:
- Loaded and inspected the FAIR² mass spectrometry dataset via Croissant
- Explored available record sets and fields, referencing by `@id`
- Filtered and normalized a key numeric field, and performed simple grouping
- Visualized distributions and group means

This notebook provides a template for analyzing rich scientific datasets using the [`mlcroissant`](https://github.com/mlcommons/croissant) interface and can be expanded for downstream biomarker discovery, batch effect analysis, or more sophisticated processing as needed.